<a href="https://colab.research.google.com/github/andresalmanzal/tfm-siniestros-bogota/blob/main/notebooks/02_enriquecimiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02 - Enriquecimiento con tablas relacionales
**Autor:** Ovidio Almanza Ledesma

## Objetivo
Incorporar a la base del notebook 01 las variables que más discriminan la gravedad, desde las tablas relacionales del conjunto *Siniestros Viales Consolidados* (Excel con hojas: SINIESTROS, ACTOR_VIAL, VEHICULOS, HIPOTESIS, DICCIONARIO). Licencia CC BY 4.0.

## Hallazgos de estructura (descubrimiento)
- **Llave de unión: `CODIGO_ACCIDENTE`** (no `FORMULARIO`), presente en todas las hojas y en la base del 01.
- La hoja `SINIESTROS` **sí contiene la `HORA` real del siniestro** → con el enriquecimiento **recuperamos la hora del día**, que no existía en la capa de puntos del 01.
- Tablas **uno-a-muchos**: ACTOR_VIAL ( aprox. 422 mil filas, una por persona), VEHICULOS (aprox. 372 mil, una por vehículo), HIPOTESIS (~234 mil, una por causa). Hay que **agregarlas** a nivel de `CODIGO_ACCIDENTE` antes de unir.
- La hoja `DICCIONARIO` decodifica los campos en código (CLASE de vehículo, CODIGO_CAUSA, DISENO_LUGAR, etc.).

## Aviso metodológico: evitar fuga de datos (data leakage)
La tabla `ACTOR_VIAL` incluye `ESTADO` (ILESO/HERIDO/MUERTO), que es casi la variable objetivo. **Se excluye como predictor.** Solo usamos información independiente del desenlace: **tipo de actor** (`CONDICION`), **clase de vehículo**, **causa/hipótesis**, **hora** y **diseño del lugar**.

### Carga reproducible del Excel consolidado
Caché local → descarga automática (cabecera de navegador + reintentos) → subida manual solo si todo falla. Se lee con `openpyxl` (incluido en Colab).

> **Nota de reproducibilidad (fuente):** el portal Datos Abiertos Bogotá rechaza conexiones desde IP de centros de datos (como las de Google Colab): la descarga automática termina en *connect timeout*. En un entorno local con conexión normal —p. ej. el equipo del evaluador— la descarga automática sí funciona. Para Colab se incluye carga manual de respaldo; el archivo queda en caché en `data/raw/`, por lo que solo se sube una vez por sesión.

In [1]:
import io, time
from pathlib import Path
import numpy as np
import pandas as pd
import requests

XLSX_URL = ("https://datosabiertos.bogota.gov.co/dataset/"
            "0b070626-fe5a-42dc-ae99-92601da166d9/resource/"
            "9e3add8e-3f46-42ce-9c0f-419964f6f598/download/"
            "siniestros_viales_consolidados_bogota_dc.xlsx")
XLSX_PATH = Path("data/raw/siniestros_consolidados.xlsx")
XLSX_PATH.parent.mkdir(parents=True, exist_ok=True)

def descargar_xlsx(url=XLSX_URL, path=XLSX_PATH, forzar=False, intentos=2):
    # cache local -> descarga automatica -> subida manual (Colab)
    if path.exists() and not forzar:
        print(f"[cache] {path}")
        return path
    headers = {"User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                              "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"),
               "Accept": "*/*"}
    for i in range(1, intentos + 1):
        try:
            print(f"[descarga] intento {i}/{intentos} ...")
            r = requests.get(url, headers=headers, timeout=(20, 300), allow_redirects=True)
            r.raise_for_status()
            path.write_bytes(r.content)
            print(f"[guardado] {path} ({path.stat().st_size/1e6:.1f} MB)")
            return path
        except Exception as e:
            print(f"   falló intento {i}: {type(e).__name__}")
    print("Descarga automática no disponible (p. ej. Colab). Sube el .xlsx (bájalo antes del navegador):")
    from google.colab import files  # type: ignore
    up = files.upload()
    path.write_bytes(up[next(iter(up))])
    return path

ruta = descargar_xlsx()
xls = pd.ExcelFile(ruta)
print("\nHojas encontradas:", xls.sheet_names)

[descarga] intento 1/2 ...
[guardado] data/raw/siniestros_consolidados.xlsx (49.2 MB)

Hojas encontradas: ['SINIESTROS', 'ACTOR_VIAL', 'VEHICULOS', 'HIPOTESIS', 'DICCIONARIO']


In [2]:
# Vistazo CRUDO (header=None) para confirmar que no hay filas de metadata arriba
for hoja in xls.sheet_names:
    print("=" * 75, "\nHOJA:", hoja)
    print(pd.read_excel(ruta, sheet_name=hoja, header=None, nrows=4).to_string(max_cols=15))
    print()

HOJA: SINIESTROS
                  0           1         2         3      4       5            6                             7                 8             9
0  CODIGO_ACCIDENTE       FECHA      HORA  GRAVEDAD  CLASE  CHOQUE  OBJETO_FIJO                     DIRECCION  CODIGO_LOCALIDAD  DISENO_LUGAR
1           4401438  01/01/2015  01:05:00         2      2     NaN          NaN               KR 64A-CL 2C 02                16             2
2           4401449  01/01/2015  05:50:00         2      3     NaN          NaN  AV AVENIDA DEL SUR-KR 65A 41                 7             1
3           4401430  01/01/2015  07:15:00         2      3     NaN          NaN             KR 19D-CL 62 S 02                19             1

HOJA: ACTOR_VIAL
                  0                   1           2          3       4     5          6          7
0  CODIGO_ACCIDENTE  CODIGO_ACCIDENTADO       FECHA  CONDICION  ESTADO  EDAD       SEXO   VEHICULO
1           4401447             2452576  01/01/2015  COND

In [3]:
# Columnas (header=0) + candidatos a llave de unión
LLAVES = {"FORMULARIO", "CODIGO_ACCIDENTE", "CODIGO", "ID"}
hojas = pd.read_excel(ruta, sheet_name=None)
for nombre, h in hojas.items():
    cands = [c for c in h.columns if str(c).strip().upper() in LLAVES]
    print(f"HOJA: {nombre} | {h.shape} | llaves: {cands}")
    print("  columnas:", list(h.columns))

HOJA: SINIESTROS | (196152, 10) | llaves: ['CODIGO_ACCIDENTE']
  columnas: ['CODIGO_ACCIDENTE', 'FECHA', 'HORA', 'GRAVEDAD', 'CLASE', 'CHOQUE', 'OBJETO_FIJO', 'DIRECCION', 'CODIGO_LOCALIDAD', 'DISENO_LUGAR']
HOJA: ACTOR_VIAL | (422416, 8) | llaves: ['CODIGO_ACCIDENTE']
  columnas: ['CODIGO_ACCIDENTE', 'CODIGO_ACCIDENTADO', 'FECHA', 'CONDICION', 'ESTADO', 'EDAD', 'SEXO', 'VEHICULO']
HOJA: VEHICULOS | (371605, 7) | llaves: ['CODIGO_ACCIDENTE']
  columnas: ['CODIGO_ACCIDENTE', 'FECHA', 'VEHICULO', 'CLASE', 'SERVICIO', 'MODALIDAD', 'ENFUGA']
HOJA: HIPOTESIS | (233819, 3) | llaves: ['CODIGO_ACCIDENTE']
  columnas: ['CODIGO_ACCIDENTE', 'FECHA', 'CODIGO_CAUSA']
HOJA: DICCIONARIO | (211, 4) | llaves: ['CODIGO']
  columnas: ['HOJA', 'CAMPO', 'CODIGO', 'DESCRIPCION']


### Diccionario de decodificación
La hoja `DICCIONARIO` (columnas HOJA, CAMPO, CODIGO, DESCRIPCION) traduce los campos numéricos a texto. Construimos un mapa `{(HOJA, CAMPO): {código: descripción}}` y una función `decodificar()` defensiva: si un campo no está en el diccionario, deja el código original y avisa, en vez de fallar.

In [4]:
hojas = pd.read_excel(ruta, sheet_name=None)   # dict {hoja: DataFrame}, header=0

dicc = hojas["DICCIONARIO"].copy()
dicc.columns = [str(c).strip().upper() for c in dicc.columns]
dicc["HOJA"]   = dicc["HOJA"].astype(str).str.strip().str.upper()
dicc["CAMPO"]  = dicc["CAMPO"].astype(str).str.strip().str.upper()
dicc["CODIGO"] = pd.to_numeric(dicc["CODIGO"], errors="coerce")

DECODE = {}
for (h, c), gdf in dicc.dropna(subset=["CODIGO"]).groupby(["HOJA", "CAMPO"]):
    DECODE[(h, c)] = dict(zip(gdf["CODIGO"].astype(int),
                              gdf["DESCRIPCION"].astype(str).str.strip()))

def decodificar(serie, hoja, campo):
    m = DECODE.get((hoja.upper(), campo.upper()))
    if not m:
        print(f"  [aviso] sin diccionario para {hoja}.{campo}; se dejan los códigos")
        return serie
    return pd.to_numeric(serie, errors="coerce").map(m)

print("Pares (HOJA, CAMPO) disponibles en el diccionario:")
for k in sorted(DECODE): print("  ", k, "->", len(DECODE[k]), "códigos")

Pares (HOJA, CAMPO) disponibles en el diccionario:
   ('HIPOTESIS', 'CODIGO_CAUSA') -> 108 códigos
   ('SINIESTROS', 'CHOQUE') -> 4 códigos
   ('SINIESTROS', 'CLASE') -> 7 códigos
   ('SINIESTROS', 'CODIGO_LOCALIDAD') -> 20 códigos
   ('SINIESTROS', 'DISENO_LUGAR') -> 13 códigos
   ('SINIESTROS', 'GRAVEDAD') -> 3 códigos
   ('SINIESTROS', 'OBJETO_FIJO') -> 11 códigos
   ('VEHICULOS', 'CLASE') -> 28 códigos
   ('VEHICULOS', 'MODALIDAD') -> 12 códigos
   ('VEHICULOS', 'SERVICIO') -> 5 códigos


### Base del notebook 01 (auto-suficiente)
Para que el 02 sea ejecutable por sí solo, `cargar_base()` intenta leer el artefacto `data/processed/01_siniestros_clean.parquet`; si no existe (p. ej. sesión nueva), lo **reconstruye** descargando la capa de puntos (API ArcGIS, que sí funciona en Colab) y aplicando el mismo saneamiento del 01.

In [5]:
ARCGIS_URL = ("https://services2.arcgis.com/NEwhEo9GGSHXcRXV/arcgis/rest/"
              "services/HistoricoSiniestros/FeatureServer/0/query")
BASE_PARQUET = Path("data/processed/01_siniestros_clean.parquet")
BASE_PARQUET.parent.mkdir(parents=True, exist_ok=True)

import unicodedata
def _norm_txt(serie):
    def _n(x):
        if pd.isna(x): return np.nan
        s = unicodedata.normalize("NFKD", str(x).strip().upper())
        s = "".join(ch for ch in s if not unicodedata.combining(ch))
        s = " ".join(s.split())
        return s if s else np.nan
    return serie.map(_n)

def _descargar_puntos(url=ARCGIS_URL, page=2000):
    regs, off = [], 0
    while True:
        params = {"where":"1=1","outFields":"*","returnGeometry":"true","outSR":4326,
                  "f":"json","resultOffset":off,"resultRecordCount":page,"orderByFields":"OBJECTID"}
        r = requests.get(url, params=params, timeout=60); r.raise_for_status()
        feats = r.json().get("features", [])
        if not feats: break
        for ft in feats: regs.append(dict(ft.get("attributes", {})))
        off += len(feats)
        if len(feats) < page: break
    return pd.DataFrame(regs)

def cargar_base():
    if BASE_PARQUET.exists():
        print(f"[base cache] {BASE_PARQUET}")
        return pd.read_parquet(BASE_PARQUET)
    print("[base] reconstruyendo desde la API de puntos ...")
    d = _descargar_puntos()
    MAP_G = {"SOLO DANOS":"Solo daños","CON HERIDOS":"Con heridos","CON MUERTOS":"Con muertos"}
    d["gravedad"] = _norm_txt(d["GRAVEDAD"]).map(MAP_G)
    d["es_grave"] = (d["gravedad"] == "Con muertos").astype("Int64")
    d["CLASE_ACC_norm"]  = _norm_txt(d["CLASE_ACC"])
    d["LOCALIDAD_norm"]  = _norm_txt(d["LOCALIDAD"])
    d["fecha_hora"] = pd.to_datetime(pd.to_numeric(d["FECHA_HORA_ACC"], errors="coerce"), unit="ms", errors="coerce")
    d["anio"] = pd.to_numeric(d["ANO_OCURRENCIA_ACC"], errors="coerce").fillna(d["fecha_hora"].dt.year).astype("Int64")
    d["mes"] = d["fecha_hora"].dt.month
    d["dia_semana"] = d["fecha_hora"].dt.dayofweek
    d["lat"] = pd.to_numeric(d["LATITUD"], errors="coerce")
    d["lon"] = pd.to_numeric(d["LONGITUD"], errors="coerce")
    cols = ["OBJECTID","CODIGO_ACCIDENTE","gravedad","es_grave","CLASE_ACC_norm","LOCALIDAD_norm",
            "fecha_hora","anio","mes","dia_semana","lat","lon","DIRECCION","CIV"]
    d = d[[c for c in cols if c in d.columns]].copy()
    try: d.to_parquet(BASE_PARQUET, index=False)
    except Exception: d.to_csv(BASE_PARQUET.with_suffix(".csv"), index=False)
    return d

base = cargar_base()
base["CODIGO_ACCIDENTE"] = pd.to_numeric(base["CODIGO_ACCIDENTE"], errors="coerce").astype("Int64")
print("Base:", base.shape, "| CODIGO_ACCIDENTE únicos:", base["CODIGO_ACCIDENTE"].nunique())

[base] reconstruyendo desde la API de puntos ...
Base: (209861, 14) | CODIGO_ACCIDENTE únicos: 209861


### SINIESTROS → recuperar la hora del día y el diseño del lugar
**Hallazgo clave:** la hoja `SINIESTROS` trae `HORA` real. La parseamos a hora entera y derivamos una **franja horaria**. También decodificamos `DISENO_LUGAR` (tipo de lugar: intersección, tramo de vía, glorieta…), un predictor útil y sin fuga.

> **Hallazgos (hora recuperada):** la hora se recuperó al 100 % en los siniestros con match. Por franja: **Tarde 70.566, Mañana 64.348, Noche 45.781, Madrugada 15.457**. La tarde concentra el mayor volumen y la madrugada el menor. Recuperar esta dimensión - imposible con la capa de puntos del 01 - permite cruzar volumen vs. letalidad por franja horaria en el modelo y el mapa.

In [6]:
sin = hojas["SINIESTROS"].copy()
sin["CODIGO_ACCIDENTE"] = pd.to_numeric(sin["CODIGO_ACCIDENTE"], errors="coerce").astype("Int64")

# Recuperar hora del día desde el texto HH:MM:SS
hora_dt = pd.to_datetime(sin["HORA"].astype(str).str.strip(), format="%H:%M:%S", errors="coerce")
sin["hora"] = hora_dt.dt.hour

def franja(h):
    if pd.isna(h): return np.nan
    h = int(h)
    return "Madrugada" if h < 6 else "Mañana" if h < 12 else "Tarde" if h < 18 else "Noche"
sin["franja_horaria"] = sin["hora"].map(franja)

sin["diseno_lugar"] = decodificar(sin["DISENO_LUGAR"], "SINIESTROS", "DISENO_LUGAR").astype("string")

sin_feats = sin[["CODIGO_ACCIDENTE", "hora", "franja_horaria", "diseno_lugar"]].drop_duplicates("CODIGO_ACCIDENTE")
print("Hora recuperada -> % no nula:", round(sin_feats["hora"].notna().mean()*100, 1))
print(sin_feats["franja_horaria"].value_counts(dropna=False))
sin_feats.head()

Hora recuperada -> % no nula: 100.0
franja_horaria
Tarde        70566
Mañana       64348
Noche        45781
Madrugada    15457
Name: count, dtype: int64


,CODIGO_ACCIDENTE,hora,franja_horaria,diseno_lugar
0,4401438,1,Madrugada,Interseccion
1,4401449,5,Madrugada,Tramo de Via
2,4401430,7,Mañana,Tramo de Via
3,4401453,9,Mañana,Tramo de Via
4,4401423,9,Mañana,Interseccion


### VEHICULOS → agregación por siniestro
Pasamos de una fila por vehículo a una fila por siniestro: número de vehículos y banderas de tipos relevantes (moto, bici, bus, camión) decodificando `CLASE`. Las motos son clave en la letalidad vial de Bogotá.

> **Hallazgos (vehículos):** se decodificaron las clases reales (automóvil, motocicleta, camioneta, bus, bicicleta, camión/furgón, volqueta, tractocamión, bicitaxi, cuatrimoto…). Las banderas resumen por siniestro la presencia de moto, bici, bus y camión, tipos clave para explicar la gravedad.

In [7]:
veh = hojas["VEHICULOS"].copy()
veh["CODIGO_ACCIDENTE"] = pd.to_numeric(veh["CODIGO_ACCIDENTE"], errors="coerce").astype("Int64")
veh["clase_desc"] = decodificar(veh["CLASE"], "VEHICULOS", "CLASE").astype("string").str.upper()

veh["es_moto"]   = veh["clase_desc"].str.contains("MOTO", na=False)
veh["es_bici"]   = veh["clase_desc"].str.contains("BICI", na=False)
veh["es_bus"]    = veh["clase_desc"].str.contains("BUS", na=False)
veh["es_camion"] = veh["clase_desc"].str.contains("CAMION", na=False)

veh_agg = veh.groupby("CODIGO_ACCIDENTE").agg(
    n_vehiculos=("VEHICULO", "count"),
    involucra_moto=("es_moto", "max"),
    involucra_bici=("es_bici", "max"),
    involucra_bus=("es_bus", "max"),
    involucra_camion=("es_camion", "max"),
).reset_index()
print("Clases de vehículo encontradas:", veh["clase_desc"].dropna().unique()[:15])
veh_agg.head()

Clases de vehículo encontradas: <StringArray>
[     'AUTOMOVIL',       'MICROBUS',    'MOTOCICLETA',            'BUS',
        'CAMPERO',      'CAMIONETA',      'BICICLETA', 'CAMION, FURGON',
       'VOLQUETA',         'BUSETA',   'TRACTOCAMION',      'MOTOCARRO',
      'MOTOCICLO',       'BICITAXI',     'CUATRIMOTO']
Length: 15, dtype: string


,CODIGO_ACCIDENTE,n_vehiculos,involucra_moto,involucra_bici,involucra_bus,involucra_camion
0,4401419,1,False,False,True,False
1,4401420,2,False,False,False,False
2,4401421,1,False,False,False,True
3,4401422,2,False,True,False,False
4,4401423,2,False,False,False,False


### ACTOR_VIAL → agregación por siniestro (sin `ESTADO`)
Banderas por **tipo de actor** (`CONDICION`: peatón, motociclista, ciclista…) y conteos, más la edad mínima/promedio de los implicados. **`ESTADO` (ileso/herido/muerto) se excluye deliberadamente** por ser fuga de datos.

> **Hallazgos (actor vial):** los tipos de actor son CONDUCTOR, PEATON, MOTOCICLISTA, PASAJERO/ACOMPAÑANTE y CICLISTA. Se generan banderas por tipo y la edad mínima/promedio de los implicados. **`ESTADO` (ileso/herido/muerto) se excluyó por ser fuga de datos.**

In [8]:
act = hojas["ACTOR_VIAL"].copy()
act["CODIGO_ACCIDENTE"] = pd.to_numeric(act["CODIGO_ACCIDENTE"], errors="coerce").astype("Int64")
act["CONDICION"] = act["CONDICION"].astype("string").str.strip().str.upper()
act["EDAD"] = pd.to_numeric(act["EDAD"], errors="coerce")

act["es_peaton"]       = act["CONDICION"].str.contains("PEATON", na=False)
act["es_motociclista"] = act["CONDICION"].str.contains("MOTO", na=False)
act["es_ciclista"]     = act["CONDICION"].str.contains("CICLISTA", na=False)
act["es_pasajero"]     = act["CONDICION"].str.contains("PASAJERO", na=False)

act_agg = act.groupby("CODIGO_ACCIDENTE").agg(
    n_actores=("CODIGO_ACCIDENTADO", "count"),
    involucra_peaton=("es_peaton", "max"),
    involucra_motociclista=("es_motociclista", "max"),
    involucra_ciclista=("es_ciclista", "max"),
    involucra_pasajero=("es_pasajero", "max"),
    edad_min=("EDAD", "min"),
    edad_prom=("EDAD", "mean"),
).reset_index()
print("Tipos de actor (CONDICION):", act["CONDICION"].dropna().unique()[:15])
act_agg.head()

Tipos de actor (CONDICION): <StringArray>
['CONDUCTOR', 'PEATON', 'MOTOCICLISTA', 'PASAJERO/ACOMPAÑANTE', 'CICLISTA']
Length: 5, dtype: string


,CODIGO_ACCIDENTE,n_actores,involucra_peaton,involucra_motociclista,involucra_ciclista,involucra_pasajero,edad_min,edad_prom
0,4401419,2,False,False,False,True,36.0,44.500000
1,4401420,2,False,False,False,False,20.0,23.500000
2,4401421,3,True,False,False,False,17.0,35.666667
3,4401422,2,False,False,True,False,30.0,41.000000
4,4401423,3,True,False,False,False,19.0,31.500000


### HIPOTESIS → causa principal por siniestro
Un siniestro puede tener varias hipótesis de causa. Tomamos la **causa principal** (la más frecuente por siniestro) decodificada y el número de hipótesis. La causa precede al desenlace, así que no es fuga.

> **Hallazgos (causa):** 108 causas decodificadas. Las más frecuentes: no mantener distancia de seguridad (51.673), adelantar cerrando (32.344), desobedecer señales (24.215), semáforo en rojo (6.994); del lado peatonal, cruzar sin observar (5.441). La causa principal entra como predictor (precede al desenlace, no es fuga).

In [9]:
hip = hojas["HIPOTESIS"].copy()
hip["CODIGO_ACCIDENTE"] = pd.to_numeric(hip["CODIGO_ACCIDENTE"], errors="coerce").astype("Int64")
hip["causa_desc"] = decodificar(hip["CODIGO_CAUSA"], "HIPOTESIS", "CODIGO_CAUSA").astype("string").str.strip()

def _moda(s):
    s = s.dropna()
    return s.mode().iloc[0] if not s.mode().empty else np.nan

hip_agg = hip.groupby("CODIGO_ACCIDENTE").agg(
    n_hipotesis=("CODIGO_CAUSA", "count"),
    causa_principal=("causa_desc", _moda),
).reset_index()
print("Causas más frecuentes:")
print(hip["causa_desc"].value_counts(dropna=False).head(10))
hip_agg.head()

Causas más frecuentes:
causa_desc
No mantener distancia de seguridad (conductor en general)    51673
Otra (conductor en general)                                  36330
Adelantar cerrando (conductor en general)                    32344
Desobedecer señales (conductor en general)                   24215
Semáforo en rojo (conductor en general)                       6994
Reverso imprudente (conductor en general)                     6508
Otras (peatón)                                                6204
Cruzar sin observar (peatón)                                  5441
No respetar prelación (conductor en general)                  4376
Arrancar sin precaución (conductor en general)                4306
Name: count, dtype: Int64


,CODIGO_ACCIDENTE,n_hipotesis,causa_principal
0,4401419,1,Frenar bruscamente (conductor en general)
1,4401420,1,Adelantar cerrando (conductor en general)
2,4401421,1,Otras (peatón)
3,4401422,1,No hacer uso de señales reflectivas (ciclista ...
4,4401423,1,Desobedecer señales (conductor en general)


### Unión final y cobertura
Unimos todo a la base por `CODIGO_ACCIDENTE` con *left join* (conservamos los 209.861 siniestros del 01). Reportamos la **cobertura** de cada fuente (no todos los siniestros tienen registro relacional) y rellenamos las banderas `involucra_*` ausentes con `False`. No se elimina ninguna fila.

> **Hallazgos (cobertura):** el enriquecimiento cubre ~86 % de los 209.861 siniestros (SINIESTROS/hora 86,3 %, VEHICULOS 86,2 %, ACTOR_VIAL 86,2 %, HIPOTESIS 86,1 %). El ~14 % restante no tiene registro relacional en el consolidado (196.152 siniestros frente a 209.861 de la capa de puntos). Se conservan todos los siniestros: las banderas ausentes quedan en `False` y los conteos en `NaN`, sin borrar filas. El dataset final tiene **31 columnas**. *(El ~14 % sin enriquecer se tratará en el modelado con un indicador de "sin dato relacional".)*

In [10]:
enr = (base
       .merge(sin_feats, on="CODIGO_ACCIDENTE", how="left")
       .merge(veh_agg,  on="CODIGO_ACCIDENTE", how="left")
       .merge(act_agg,  on="CODIGO_ACCIDENTE", how="left")
       .merge(hip_agg,  on="CODIGO_ACCIDENTE", how="left"))

print("Cobertura del enriquecimiento (% de la base con dato):")
for nombre, agg in [("SINIESTROS/hora", sin_feats), ("VEHICULOS", veh_agg),
                    ("ACTOR_VIAL", act_agg), ("HIPOTESIS", hip_agg)]:
    cob = base["CODIGO_ACCIDENTE"].isin(agg["CODIGO_ACCIDENTE"]).mean() * 100
    print(f"  {nombre:16s}: {cob:5.1f}%")

flags = [c for c in enr.columns if c.startswith("involucra_")]
enr[flags] = enr[flags].fillna(False).astype(bool)

print("\nEnriquecido:", enr.shape)
print("Nuevas columnas:", [c for c in enr.columns if c not in base.columns])
enr.head(3)

Cobertura del enriquecimiento (% de la base con dato):
  SINIESTROS/hora :  86.3%
  VEHICULOS       :  86.2%
  ACTOR_VIAL      :  86.2%
  HIPOTESIS       :  86.1%

Enriquecido: (209861, 31)
Nuevas columnas: ['hora', 'franja_horaria', 'diseno_lugar', 'n_vehiculos', 'involucra_moto', 'involucra_bici', 'involucra_bus', 'involucra_camion', 'n_actores', 'involucra_peaton', 'involucra_motociclista', 'involucra_ciclista', 'involucra_pasajero', 'edad_min', 'edad_prom', 'n_hipotesis', 'causa_principal']


,OBJECTID,CODIGO_ACCIDENTE,gravedad,es_grave,CLASE_ACC_norm,LOCALIDAD_norm,fecha_hora,anio,mes,dia_semana,...,involucra_camion,n_actores,involucra_peaton,involucra_motociclista,involucra_ciclista,involucra_pasajero,edad_min,edad_prom,n_hipotesis,causa_principal
0,631773,4453111,Solo daños,0,CHOQUE,SAN CRISTOBAL,2016-08-02,2016,8,1,...,True,2.0,False,False,False,False,27.0,36.0,2.0,Desobedecer señales (conductor en general)
1,631774,4509563,Solo daños,0,CHOQUE,PUENTE ARANDA,2018-02-23,2018,2,4,...,False,2.0,False,False,False,False,47.0,48.5,1.0,No respetar prelación de intersecciones o giro...
2,631775,4461204,Solo daños,0,CHOQUE,RAFAEL URIBE URIBE,2016-10-18,2016,10,1,...,True,2.0,False,False,False,False,28.0,33.0,1.0,No respetar prelación (conductor en general)


### Persistencia del artefacto para el notebook 03
Guardamos el dataset enriquecido en `data/processed/02_siniestros_enriquecido.parquet` (fallback CSV). Será el insumo del **notebook 03 (features adicionales)**: ingeniería espacial (mallas/zonas), interacciones y preparación del dataset modelable.

In [11]:
destino = Path("data/processed/02_siniestros_enriquecido.parquet")
destino.parent.mkdir(parents=True, exist_ok=True)
try:
    enr.to_parquet(destino, index=False)
    print(f"Guardado: {destino} | {enr.shape}")
except Exception as e:
    destino = destino.with_suffix(".csv")
    enr.to_csv(destino, index=False)
    print(f"pyarrow no disponible ({e}). Guardado CSV: {destino} | {enr.shape}")

Guardado: data/processed/02_siniestros_enriquecido.parquet | (209861, 31)


### Resumen y siguiente paso
- Unimos las tablas relacionales por **`CODIGO_ACCIDENTE`**, agregando a nivel de siniestro.
- **Recuperamos la hora del día** (perdida en la capa de puntos) y la franja horaria desde el Excel consolidado.
- Nuevas señales sin fuga: tipo de vehículo (moto/bici/bus/camión), tipo de actor (peatón/motociclista/ciclista), edad de implicados, causa principal y diseño del lugar.
- **`ESTADO` (condición de la víctima) excluido** por ser fuga de datos.
- Cobertura reportada y banderas ausentes tratadas; sin borrar filas.

**Notebook 03 — Features adicionales:** ingeniería espacial (agregación por zona/malla), interacciones (p. ej. moto × franja horaria) y consolidación del dataset modelable.